# IS630: Green Light for an Indie Studio
## Notebook 04: AQ3, What Factors Seem to Be the Most Important for a Game?
**Group**: [Insert names]

> **AI Coding Assistant Declaration**:
> An AI coding assistant drafted this specification at the group's direction. The group must review and accept it before code is written. All interpretations, evaluations of model performance and conclusions will be written by group members in the designated cells.

**AQ3** (group wording): *What factors seem to be the most important for a game? We want to study if there are certain factors that are especially important that determine the success of a game, and highlight ones that a studio can control.*

## Specification
*Written by AI Assistant*

Status: spec only, no code yet. Success is the log completion ratio, as in notebook 03. AQ3 reuses notebook 03's sample, predictors, standardisation, OLS model and Bayesian model, so a factor's importance here is read off the same fits that AQ2 evaluated.

**Reads:** `data/processed/02_train.parquet`, `data/processed/02_validation.parquet`, `data/processed/02_feature_spec.json`, `outputs/models/03_aq2_bayes_main.nc` (refit if missing, it is gitignored), `outputs/tables/03_aq2_influence_refits.csv`, `outputs/tables/03_aq2_vif.csv`
**Writes:** `data/interim/04_aq3_importance.parquet` for AQ4, tables `outputs/tables/04_aq3_*.csv`, figures `outputs/figures/04_aq3_*.png`

The test holdout is never loaded. Every result below gets a blank interpretation cell.

### 1. Factor map
- One row per predictor from notebook 02: the column, a plain-English name, a factor block, and whether a studio controls it before launch.
- Blocks group columns that describe one decision, so that one-hot sets are judged together: price (`feature_log_price`, `flag_free_to_play`), release timing (`flag_q4_release`), platforms (`steam_mac`, `steam_linux`), localisation (language and audio counts, `flag_has_full_audio`), store features (achievements, controller, cloud, trading cards, VR, co-op, multiplayer), genre (the `genre_*` flags), content rating (`flag_age_restricted`), publishing (`flag_self_published`), storefront (`flag_has_website`, `flag_has_support_contact`), and not controllable (`steam_years_since_release`, `feature_log_dev_game_count`, `flag_missing_genre`).
- The controllable column is a draft for the group to confirm (see decisions below).
- Output: `04_aq3_factor_map.csv`.

### 2. Effect sizes on a common scale
- Continuous coefficients in notebook 03 are per SD and binary ones are per 0 to 1 switch, so they are not directly comparable. Rescale continuous predictors to per 2 SD, which puts them on roughly the same footing as a balanced binary.
- For each predictor report: OLS estimate with clustered 95% CI and Benjamini-Hochberg q, Bayesian posterior mean with 95% credible interval, posterior probability of the sign, and the completion-ratio multiplier `exp(b)`.
- Forest plot ordered by Bayesian posterior mean, coloured by block, with controllable factors marked.
- Output: `04_aq3_effects.csv`, `04_aq3_effects.png`.

### 3. Out-of-sample block importance
- Drop one block at a time, refit OLS on development training, score on validation. Importance is the rise in validation RMSE over the full model.
- 95% interval from the same developer-level bootstrap as notebook 03, 2,000 resamples. A block whose interval sits above zero adds predictive information that the other blocks do not carry.
- This measure handles collinear blocks better than coefficient size: a block that overlaps another loses little when dropped, whatever its coefficient.
- Output: `04_aq3_block_importance.csv`, `04_aq3_block_importance.png`.

### 4. Permutation importance per predictor
- On validation, shuffle one predictor at a time in the fitted OLS and Bayesian models, 200 repeats, and record the rise in RMSE.
- Per-column view to set beside the block view. Shuffling a collinear column understates it, so read alongside the VIF table from notebook 03.
- Output: `04_aq3_permutation.csv`.

### 5. Agreement and robustness
- One table with each predictor's rank under each measure: absolute effect (OLS, Bayesian), block importance, permutation importance. Kendall's tau between the rankings.
- Add two robustness flags from notebook 03: whether the predictor's interval excludes zero in both refits from the influence diagnostics, and whether its VIF exceeds 5.
- Output: `04_aq3_agreement.csv`.

### 6. Reduced model
- Keep the blocks that pass a rule fixed before looking at the result: the block-importance interval in section 3 excludes zero. If no block passes, report that and skip the reduced model.
- Refit OLS and the Bayesian model on the kept blocks. Compare with the full model and the baseline on validation with the same metrics and bootstrap as notebook 03.
- Output: `04_aq3_reduced_vs_full.csv`.

### 7. Effects a studio can read
- For the controllable factors with the clearest effects, show the predicted completion ratio on the original scale, holding other predictors at their training means or modes.
- Continuous factors (price, language count, achievements): a curve against the raw value, with the 90% Bayesian credible band. Binary factors: the predicted ratio with and without, with interval.
- Price is the price at scrape time, not the launch price. This caveat goes beside every price result.
- Output: `04_aq3_partial_effects.png`.

### Save
- `04_aq3_importance.parquet`: one row per predictor with its block, controllable flag, effect, interval, ranks and robustness flags, for AQ4.

### Limits to state in the notebook
- Observational data: an effect is an association within story-driven, HowLongToBeat-matched games with at least 50 reviews and 5 polls, not a causal effect of changing the factor.
- If notebook 03 finds little predictive signal, the importance measures will be small and noisy, and that is itself the result for AQ3.
- SteamSpy playtime, the numerator, is an estimate with no sample size behind it.

### Decisions for the group before code is written
1. Confirm or change the controllable column in the factor map, in particular `flag_self_published` and `steam_years_since_release`.
2. Confirm the per 2 SD rescaling in section 2, or keep per 1 SD.
3. Confirm the reduced-model rule in section 6.
4. Decide whether permutation importance (section 4) is worth the space, or whether blocks are enough.